# Price curves (commodity forward prices)

Deep-dive reference: **forward price** term structures as a function of time (e.g. energy or metals).

## Concept

A **price curve** stores **forward prices** (or levels) at future **year fractions** from a base date. Futures markets quote prices at **contract expiries**; in code you typically map those to **time-to-maturity** on a chosen day-count. Use **`price(t)`** to interpolate the forward level.

## API walkthrough

`PriceCurve(id, base_date, knots)` with knots **`(time_years, forward_price)`** — not calendar dates in the constructor; dates are implied via your mapping into year fractions.

In [ ]:
try:
    from datetime import date

    from finstack.core.market_data import PriceCurve
except ImportError as exc:
    print("PriceCurve is not importable from finstack.core.market_data:", exc)
else:
    base = date(2024, 1, 1)
    # Stylized WTI-style forward curve: (time_years, USD/bbl)
    wti = PriceCurve(
        "WTI",
        base,
        [(0.0, 72.0), (0.25, 71.5), (0.5, 71.0), (1.0, 70.0), (2.0, 68.5)],
        day_count="act_365f",
    )
    print("curve:", wti)
    for t in (0.0, 1 / 12, 0.5, 1.0, 1.75):
        print(f"forward price at t={t:.3f}y -> {wti.price(t):.2f} USD/bbl")

## Practical example

**Backwardation vs contango:** compare **near** vs **far** forwards from the same curve.

In [ ]:
try:
    from datetime import date

    from finstack.core.market_data import PriceCurve
except ImportError as exc:
    print("Skip example — PriceCurve unavailable:", exc)
else:
    base = date(2024, 1, 1)
    backwardated = PriceCurve("WTI-BACK", base, [(0.0, 75.0), (1.0, 70.0)], day_count="act_365f")
    contango = PriceCurve("WTI-CONT", base, [(0.0, 70.0), (1.0, 75.0)], day_count="act_365f")
    print("Backwardated: front vs 1Y =", backwardated.price(0.0), backwardated.price(1.0))
    print("Contango:     front vs 1Y =", contango.price(0.0), contango.price(1.0))

## Takeaways

- **`price(t)`** is the interpolated **forward price** at horizon `t` in **years**.
- Calendar **pillar dates** must be converted to **year fractions** consistently with **`day_count`** and your **base_date**.
- For **rate indices** (SOFR, etc.) prefer **`ForwardCurve`**; **`PriceCurve`** is tailored to **price levels**.